# Tanzania Financial News Scraper
### DS & AI Year I: Data Collection Notebook

This notebook collects financial news headlines from major Tanzanian news outlets
for the period **2022 – 2024**. The collected dataset will be used for:

- Sentiment analysis of financial news
- Correlation analysis with TZS/USD exchange rates and inflation data

**Target outlets:**
- Daily News Tanzania (`dailynews.co.tz`)
- The Citizen Tanzania (`thecitizen.co.tz`)

**Authors:** *(add your names)*  
**Date:** *(add date)*

---

> **Before running:** Make sure you have installed the required libraries.  
> Run the cell in **Section 1** first if this is your first time.


## Section 1: Requirements & Imports

Run the install cell once if you don't have the libraries yet.  
After that you only need to run the imports cell every session.


In [ ]:
# ── Install once (comment out after first run) ──
# !pip install requests beautifulsoup4

In [1]:
# ── Imports (run every session) ──
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

print("All libraries loaded successfully")
print("kesho debate")

All libraries loaded successfully
kesho debate


## Section 2: How BeautifulSoup Works

Before scraping real websites, let's understand the core tool we're using:
**BeautifulSoup (bs4)**.

A webpage is just HTML, a tree of nested tags. BeautifulSoup lets us
navigate and extract from that tree using CSS selectors, tag names, or attributes.


### a.) Parsing hardcoded HTML (no internet needed)

Below is a tiny fake news page. We'll extract headlines and dates from it
exactly the same way we will from real sites.

In [7]:
# Fake HTML that looks like a news listing page
fake_html = """
<html>
  <body>
    <div class="news-list">

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/1">Shilling strengthens against the dollar</a>
        </h2>
        <span class="date">2023-04-12</span>
      </div>

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/2">Inflation drops to 3.8 percent in March</a>
        </h2>
        <span class="date">2023-04-10</span>
      </div>

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/3">Tanzania secures new Chinese FDI deal</a>
        </h2>
        <span class="date">2023-04-08</span>
      </div>

    </div>
  </body>
</html>
"""

# Parse it
soup = BeautifulSoup(fake_html, 'html.parser')

# Extract all articles
articles = soup.select('div.article')
print(f"Found {len(articles)} articles\n")

for article in articles:
    headline = article.select_one('h2.headline a').get_text(strip=True)
    date     = article.select_one('span.date').get_text(strip=True)
    link     = article.select_one('h2.headline a')['href']
    print(f" {date} | {headline}")
    print(f" Link: {link}\n")


Found 3 articles

 2023-04-12 | Shilling strengthens against the dollar
 Link: Thek2news/article/1

 2023-04-10 | Inflation drops to 3.8 percent in March
 Link: Thek2news/article/2

 2023-04-08 | Tanzania secures new Chinese FDI deal
 Link: Thek2news/article/3



**What just happened?**

| Code | What it does |
|------|-------------|
| `BeautifulSoup(html, 'html.parser')` | Parses the raw HTML string into a navigable tree |
| `soup.select('div.article')` | Finds ALL elements matching the CSS selector |
| `soup.select_one('h2.headline a')` | Finds the FIRST matching element |
| `.get_text(strip=True)` | Extracts the visible text, removing whitespace |
| `['href']` | Reads an HTML attribute value |

This is the entire foundation of web scraping everything else is just applying
these same ideas to real websites.


### b.) Scraping a live website

Now the same idea but on a real URL. We use `requests` to download the page HTML,
then BeautifulSoup to parse it.

We use **`books.toscrape.com`** — a site built specifically for scraping practice.
It has no login, no blocks, and a clean predictable structure. Perfect for learning.


In [9]:
# Download the page
url = "http://books.toscrape.com/"
response = requests.get(url, timeout=10) # timeout is how long the fetxh is going to keep ontrying to receive the site if exceeded returns error

print(f"Status code : {response.status_code}")   # 200 = success
print(f"Content size: {len(response.text):,} characters of HTML")


Status code : 200
Content size: 51,294 characters of HTML


In [10]:
# Parse and extract book titles + prices
soup = BeautifulSoup(response.text, 'html.parser')

books = soup.select('article.product_pod')
print(f"Found {len(books)} books on this page\n")

for book in books[:5]:   # show first 5 only
    title = book.select_one('h3 a')['title']
    price = book.select_one('p.price_color').get_text(strip=True)
    print(f" Book Name:  {title}")
    print(f" Book Price: {price}\n")


Found 20 books on this page

 Book name:  A Light in the Attic
 Book Price: Â£51.77

 Book name:  Tipping the Velvet
 Book Price: Â£53.74

 Book name:  Soumission
 Book Price: Â£50.10

 Book name:  Sharp Objects
 Book Price: Â£47.82

 Book name:  Sapiens: A Brief History of Humankind
 Book Price: Â£54.23



---
### Understanding the Web Scraping Code

`BeautifulSoup(...)` parses the HTML page.

```python
soup.select('article.product_pod')
```

This finds all `<article>` elements with class `product_pod`, where each represents a book.

So `books` becomes a list of all book elements.

---

##### Looping Through Books

```python
books[:5]
```

This limits the loop to only the first **5 books**.

Useful for:

* Testing
* Previewing results
* Avoiding excessive output

---

##### Extracting the Title

```python
title = book.select_one('h3 a')['title']
```

This:

* Finds the `<a>` tag inside `<h3>`
* Reads its `title` attribute

Example:

```html
<a title="A Light in the Attic">
```

Result:

```python
title = "A Light in the Attic"
```

---

##### Extracting the Price

```python
price = book.select_one('p.price_color').get_text(strip=True)
```

This finds the price element and extracts its text.

Example:

```html
<p class="price_color">£53.74</p>
```

Result:

```python
price = "£53.74"
```

The scraper now extracts both the **book title** and **price**.


**Key difference from the hardcoded demo:**

We added one step — `requests.get(url)` — which downloads the raw HTML from the internet.
After that, BeautifulSoup works exactly the same way.

This is the complete scraping workflow:
```
URL → requests.get() → raw HTML → BeautifulSoup() → extract data → save
```

Everything in the rest of this notebook is just this loop, applied to Tanzanian news sites.


# Okaseni Butchery — Project Planning

## Mission
Build a fast, mobile-first business website for Okaseni Butchery, Dar es Salaam,
Tanzania. The site must work on slow mobile connections, support English and Swahili
via a toggle, and allow customers to place orders directly through WhatsApp. The
business owner manages all product content through a simple admin interface backed
by Firebase Firestore — no developer needed for day-to-day updates.

---

## Current Status
- [ ] index.html — complete
- [ ] products.html — not started
- [ ] order.html — not started
- [ ] contact.html — not started
- [ ] admin.html — not started
- [ ] css/style.css — not started
- [ ] js/main.js — not started
- [ ] js/products.js — not started
- [ ] js/order.js — not started
- [ ] js/admin.js — not started
- [ ] Firebase Firestore — DONE (test data added)
- [ ] .gitignore — DONE
- [ ] js/firebase-config.js — DONE

Update the checkboxes as each file is completed.

---

## Build Order
Follow this sequence strictly. Do not skip ahead or build a later page before
the previous one is reviewed and checked off.

1. CSS foundation + navbar
2. index.html — homepage
3. products.html — product listing
4. contact.html — location and hours
5. order.html — order form
6. admin.html — owner product management
7. Language toggle system
8. Final pass — performance, links, mobile check

---

## Page-by-Page Specifications

### 1. css/style.css + navbar (first task)
Goal: establish the entire design system before any page content is built.

Must include:
- CSS reset (box-sizing, margin, padding, font inheritance)
- All six CSS custom properties defined on :root
- Base typography: Oswald for headings, Lato for body, loaded via Google Fonts
  link in every HTML head — not @import in CSS
- Navbar: sticky, 64px height, charcoal background, logo left, links right,
  EN/SW toggle far right, hamburger for mobile
- Hamburger toggle logic in main.js — no inline onclick handlers in HTML
- Active page link highlighted in --color-gold

Done when: navbar renders correctly on mobile and desktop, all CSS variables
are defined, fonts load, hamburger opens and closes the mobile menu.

---

### 2. index.html — Homepage
Goal: communicate the brand, build trust, push visitors toward ordering.

Sections in order:
- Navbar (shared component, same across all pages)
- Hero: full viewport height, dark overlay, H1 tagline, subtitle, two buttons
  (primary CTA to order.html, ghost button to products.html)
- Why Okaseni: three value proposition cards — Fresh Daily, Dar es Salaam
  Delivery, Bei Nzuri. Icons or simple visual, short text each.
- Featured products: three hardcoded cards (Fillet, T-Bone, Sangara) — these
  are static on the homepage only, not pulled from Firebase. A "See All" link
  goes to products.html.
- How to order: three simple steps — Browse, Fill the form, Receive delivery.
  Visual step indicator.
- Footer (shared component, same across all pages)

Done when: all sections visible, mobile layout correct, CTA buttons link to
correct pages, no broken images or fonts.

---

### 3. products.html — Product Listing
Goal: show all available products dynamically from Firebase, grouped by category.

Sections in order:
- Navbar
- Page header: title and short description
- Category filter tabs: All / Premium Cuts / Specialty / Fish
  Clicking a tab filters the visible cards without reloading the page.
- Product grid: cards rendered by products.js from Firestore
  Each card: image, category badge, name, description, price per kg, 
  order button that links to order.html
- If no products load: show a friendly empty state message, not a blank page
- Footer

Firebase connection:
- products.js reads the "products" collection
- unit: string — "kg" or "pieces", default "kg" for all current products
- Filters out documents where available is false
- Groups results by category field
- Renders cards into the grid container
- Category tab click filters already-loaded cards in memory — no extra
  Firebase reads on tab switch

Done when: products load from Firebase, category filter works, unavailable
products are hidden, empty state shows if collection is empty.

---

### Cart System (cart.js)
A localStorage-based cart shared across products.html and order.html.
localStorage key: "okaseni_cart"
Data structure: array of cart item objects:
{ id, name, category, price, unit, quantity }
cart.js exposes these functions:

getCart() → returns parsed array from localStorage
addToCart(product, quantity) → adds or updates item
removeFromCart(id) → removes item by Firestore document id
updateQuantity(id, quantity) → updates quantity for existing item
clearCart() → empties the cart completely
getCartCount() → returns total number of items in cart

Cart sidebar (rendered by cart.js):

Fixed position, slides in from right on cart icon click
Width: 320px desktop, 100% mobile
Shows: item name, quantity + unit, remove button per item
Does NOT show prices or totals — clean and simple
Footer of sidebar: item count, Checkout button → order.html
Overlay behind sidebar dims the page, clicking it closes sidebar
Cart icon in navbar shows a red badge with item count when cart > 0

Unit handling:

"kg" products: quantity is float, step 0.5, min 0.5, displayed as "1.5 kg"
"pieces" products: quantity is integer, step 1, min 1, displayed as "3 pieces"

---

### 4. contact.html — Contact and Location
Goal: give customers every way to reach the business.

Sections in order:
- Navbar
- Page header
- Two column layout on desktop, stacked on mobile:
  Left: business details — address (Boko Basiahaya, Dar es Salaam), hours
  (Daily 6:00 AM – 8:00 PM), phone numbers as clickable tel: links,
  WhatsApp buttons for both numbers, Instagram link
  Right: Google Maps embed for Boko Basiahaya area
- A prominent "Place an Order" CTA banner linking to order.html
- Footer

No Firebase needed on this page.

Done when: all contact details correct, WhatsApp links open correctly, map
loads, CTA links to order.html.

---

### 5. order.html — Checkout Page
Goal: let customers review their cart and send their order via WhatsApp.
This page does NOT handle product selection.
Product selection happens on products.html via the cart system.
Sections in order:

Navbar
Page header: "Your Order"
Cart summary: rendered from localStorage "okaseni_cart"
Read-only list — name, quantity, unit, price per unit if > 0,
line total if price > 0
Empty cart state: friendly message + link back to products.html
Checkout form: name, phone, delivery address, optional notes
Submit: builds WhatsApp message, opens wa.me link, shows success,
clears cart, disables button
Footer

Firebase: NOT loaded on this page. Cart comes from localStorage only.
Done when: cart summary renders correctly from localStorage, form
validates, WhatsApp message is correctly formatted with emoji,
totals only appear when prices exist, cart clears after send.

---

### 6. admin.html — Owner Product Management
Goal: let the business owner manage products without touching code.

Access control:
- On page load check localStorage for an authenticated session
- If not authenticated show only a centered PIN entry screen, hide everything else
- PIN verified in admin.js against a stored constant
- Correct PIN: set session in localStorage, reveal admin content
- Wrong PIN: show error message, clear input

Admin interface sections:
- Header with logout button
- Add / Edit product form:
  - Product name, category (select), price (number), description (textarea
    max 120 chars), image URL (text), available (checkbox)
  - Save button: adds new document or updates existing one
  - Cancel button: clears form
- Product list table:
  - Columns: image thumbnail, name, category, price, available toggle, edit, delete
  - Available toggle updates Firestore immediately on click
  - Edit pre-fills the form above with that product's data
  - Delete shows a browser confirm dialog before removing document
- Success and error messages for every Firebase operation

Firebase connection:
- admin.js reads all products including unavailable ones
- admin.js writes: addDoc, updateDoc, deleteDoc
- firebase-config.js loaded before admin.js in the HTML

Done when: owner can add a product, see it appear in the list, edit it, toggle
availability, and delete it. All changes reflected in Firestore immediately.

---

### 7. Language Toggle System
Goal: every page switches cleanly between English and Swahili.

Implementation (in main.js):
- On every page load: read language preference from localStorage
  Default to "en" if nothing stored
- setLanguage(lang) function:
  Loops over all elements with data-en and data-sw attributes
  Sets textContent to the matching attribute value
  Saves lang to localStorage
- Language toggle buttons in navbar call setLanguage() on click
- Active button styled with --color-gold

HTML pattern for every translatable element:
  data-en="English text here" data-sw="Swahili text here"

Translatable elements include:
- All nav links
- All headings and subheadings
- All button labels
- All section descriptions
- Form labels and placeholders
- Footer text
- Success and error messages

Done when: clicking EN/SW on any page switches all text, preference persists
on page reload and when navigating between pages.

---

### 8. Final Pass
Go through every page and check:

Performance:
- [ ] All images compressed and have alt attributes
- [ ] Firebase scripts only loaded on pages that need them
  (products.html, order.html, admin.html)
- [ ] No console.log statements anywhere
- [ ] Google Fonts link present in every HTML head

Links and navigation:
- [ ] All nav links work on every page
- [ ] All CTA buttons link to correct pages
- [ ] WhatsApp links use correct number format (255747471187)
- [ ] Instagram link opens @okaseni_butchery
- [ ] Tel links work on mobile

Mobile:
- [ ] Hamburger menu works on all pages
- [ ] No horizontal scroll on any page at 320px width
- [ ] Product grid readable on small screens
- [ ] Order form usable on mobile keyboard

Before going live:
- [ ] Firebase security rules locked down (not in test mode)
- [ ] firebase-config.js confirmed in .gitignore
- [ ] Test a full order flow end to end on a real phone
- [ ] Share admin PIN with business owner only

---

## Firebase Security Rules
Before going live replace the test mode rules in Firebase console with these:
```
rules_version = '2';
service cloud.firestore {
  match /databases/{database}/documents {
    match /products/{document} {
      allow read: if true;
      allow write: if false;
    }
  }
}
```

This allows anyone to read products (needed for the public site) but blocks
all writes from the browser. Admin writes will be handled separately once
proper auth is added in a future version.

Note: this means admin.html will stop working after these rules are applied.
Admin functionality requires Firebase Auth or a backend in a future phase.
For launch, the owner updates products directly in the Firebase console.
This is a known limitation of the current architecture and an acceptable
tradeoff for a first version.

---

## Business Reference

### Brand
- Name: Okaseni Butchery
- Tagline: A cut above the rest
- Location: Boko Basiahaya, Dar es Salaam, Tanzania
- Hours: Daily 6:00 AM – 8:00 PM
- Instagram: @okaseni_butchery

### Contact
- WhatsApp / Phone 1: +255 747 471 187
- WhatsApp / Phone 2: +255 740 277 321

### Product Categories
- Premium Cuts: Fillet, Steak, Sirloin, T-Bone, Ribs
- Specialty: Maini (Liver)
- Fish: Sato (Tilapia), Sangara (Nile Perch)

### Value Propositions
- Fresh meat processed daily
- Delivery throughout Dar es Salaam
- Bei nzuri — good prices

### Language Strings (key phrases)
EN: Fresh Meat, Every Day → SW: Nyama Safi, Kila Siku
EN: Our Products → SW: Bidhaa Zetu
EN: Order Now → SW: Agiza Sasa
EN: Contact Us → SW: Wasiliana Nasi
EN: Home → SW: Nyumbani
EN: Fresh Daily → SW: Safi Kila Siku
EN: Good Prices → SW: Bei Nzuri
EN: We Deliver to You → SW: Tunafikisha Kwako
EN: A cut above the rest → SW: Bora kuliko wote